# Bonus: Part 2 across the whole job postings dataset

Generative AI - Assignment 1
Darrsheni Sapovadia (27PGAI0063)

The bonus run for Part 2: all 2,277 postings instead of the first 25. Category plus the
three requirement fields, same as the graded notebook.

## Why this one runs locally

Groq is no use for a dataset this size. The free tier allows 200,000 tokens a day and this
needs well over a million, so it runs dry long before the job is done. The brief suggests
Ollama with a small local model instead, which is what this notebook uses. No rate limits,
just a slow laptop.

Three things make the full run finish in a sensible time:

- **One call per row.** The graded notebook used a separate prompt for each task. Here the
  model returns everything in one JSON object, which is a third of the work for Part 1.
- **Eight requests at once.** A 3b model does not come close to using eight cores on its own,
  so running several at a time roughly triples the throughput. Ollama needs to be started
  with `OLLAMA_NUM_PARALLEL` set to match.
- **A JSON schema rather than just asking for JSON.** This one mattered more than I expected.
  Asking politely for one of five categories, a 3b model happily answers "Art" or "Sports"
  instead, and my first attempt was getting about a third of them right. Passing a schema
  with an `enum` makes Ollama constrain the output as it generates, so an invalid category
  is not possible any more.

It is still a much smaller model than the graded notebooks use, and it is less accurate. The
brief does ask to balance inference speed against accuracy, and a 3b model at roughly ten
seconds a row is where that lands on this hardware.

## Setup

In [ ]:
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

MODEL = "llama3.2:3b"
WORKERS = 8

print("Model:", MODEL, "| parallel requests:", WORKERS)

In [ ]:
def find_json(text):
    """Grab the JSON object out of a reply and ignore anything around it."""
    found = re.search(r"\{.*\}", text, re.S)
    if not found:
        return {}
    try:
        return json.loads(found.group())
    except json.JSONDecodeError:
        return {}


def run_all(work, handle, checkpoint, every=100):
    """Run `handle` over every row, saving progress as it goes.

    This takes hours on a laptop CPU, so it writes a checkpoint every hundred
    rows. If the run dies partway through, running the cell again picks up from
    where it stopped rather than starting the whole thing over.
    """
    done = {}
    if os.path.exists(checkpoint):
        saved = pd.read_csv(checkpoint).fillna("")
        done = {int(r["row"]): r.to_dict() for _, r in saved.iterrows()}
        print(f"found a checkpoint with {len(done)} rows already done")

    todo = [w for w in work if w[0] not in done]
    print(f"{len(todo)} rows still to do")

    started_with = len(done)
    start = time.time()

    for at in range(0, len(todo), every):
        chunk = todo[at:at + every]
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            for result in pool.map(handle, chunk):
                done[result["row"]] = result

        pd.DataFrame(sorted(done.values(), key=lambda r: r["row"])).to_csv(
            checkpoint, index=False
        )
        per_row = (time.time() - start) / max(1, len(done) - started_with)
        left = (len(work) - len(done)) * per_row / 60
        print(f"  {len(done)}/{len(work)} done, roughly {left:.0f} min left")

    return pd.DataFrame(sorted(done.values(), key=lambda r: r["row"]))

## Load the whole dataset

In [ ]:
jobs = pd.read_csv("../data/job_title_des.csv")
jobs = jobs.drop(columns=["Unnamed: 0"]).rename(columns={
    "Job Title": "Job_Title",
    "Job Description": "Job_Description",
}).reset_index(drop=True)

print("Postings to process:", len(jobs))
jobs.head(3)

## The prompt and the schema

The category list is the one from the brief, and the schema pins the answer to it so the
model cannot invent a seventh domain halfway through a six hour run.

In [ ]:
DOMAINS = ["Technology/IT", "Finance", "Marketing", "Healthcare", "Education", "Others"]

SCHEMA = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": DOMAINS},
        "skills": {"type": "array", "items": {"type": "string"}},
        "education": {"type": "string"},
        "experience": {"type": "string"},
    },
    "required": ["category", "skills", "education", "experience"],
}

llm = ChatOllama(model=MODEL, temperature=0, num_predict=400, format=SCHEMA)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You read job postings and reply with JSON only."),
    ("human",
     "Categorise this job posting and pull out what it asks for.\n\n"
     "category must be exactly one of Technology/IT, Finance, Marketing, Healthcare, "
     "Education, Others.\n"
     "skills are the tools, languages and abilities the posting asks for.\n"
     "education is the minimum degree wanted, or \"Not specified\" if it does not say.\n"
     "experience is the years or level wanted, or \"Not specified\" if it does not say.\n"
     "Never invent anything that is not in the posting.\n\n"
     "Job title: {title}\n"
     "Job description:\n{description}"),
])

chain = prompt | llm | StrOutputParser()

In [ ]:
def analyse(item):
    """Categorise one posting and pull its requirements out."""
    position, title, description = item
    category, skills = "Others", ""
    education = experience = "Not specified"

    try:
        data = find_json(chain.invoke({
            "title": title,
            "description": description[:1200],
        }))

        raw = str(data.get("category", ""))
        for domain in DOMAINS:
            if domain.lower() in raw.lower():
                category = domain
                break

        found = data.get("skills", [])
        if isinstance(found, str):
            found = [found]
        skills = ", ".join(dict.fromkeys(str(s).strip() for s in found if str(s).strip()))

        education = str(data.get("education", "") or "").strip() or "Not specified"
        experience = str(data.get("experience", "") or "").strip() or "Not specified"
    except Exception as error:
        skills = f"failed: {type(error).__name__}"

    return {
        "row": position,
        "Predicted_Category": category,
        "Required_Skills": skills or "Not specified",
        "Education_Required": education,
        "Experience_Required": experience,
    }

## Run it

About six hours for all 2,277, checkpointed every hundred rows the same way.

In [ ]:
work = [(i, r["Job_Title"], r["Job_Description"]) for i, r in jobs.iterrows()]

results = run_all(work, analyse, "../outputs/bonus_part2_checkpoint.csv")

print("\nFinished:", len(results), "postings")

## Put the results back on the dataframe

In [ ]:
merged = jobs.join(results.set_index("row")[
    ["Predicted_Category", "Required_Skills", "Education_Required", "Experience_Required"]
])

pd.set_option("display.max_colwidth", 45)
merged.head(15)

In [ ]:
print("Rows:", len(merged))
print()
print("Categories across the whole dataset:")
print(merged["Predicted_Category"].value_counts().to_string())
print()
for column in ["Education_Required", "Experience_Required"]:
    missing = (merged[column] == "Not specified").sum()
    print(f"{column}: {missing} of {len(merged)} postings do not say")

The spread of categories is the interesting bit here. The first 25 rows are all developer
jobs, so the graded notebook only ever saw Technology/IT. Across the full dataset there is a
good deal more variety, which is a better test of whether the classifier is doing anything
at all or just answering the same thing every time.

## Save

In [ ]:
merged.to_csv("../outputs/bonus_part2_full_job_results.csv", index=False)

print("Saved", len(merged), "rows to outputs/bonus_part2_full_job_results.csv")